# Magic Hour | AI Voice Editor - production notebook

Upload a voice recording, pick an edit, listen to the result, download the MP3 (WAV master kept). One backend (v0.6.0) behind three features:

- **Audio enhancing (cleanup):** enhance / denoise / dereverb
- **Transcript editing:** replace / insert / delete words in what was said
- **Prompt editing:** emotion, voice style, whisper, speed, or a custom plain-English instruction

Run the steps in order: **1** install the backend, **2** optional Drive mount (only to keep files across sessions; skip it if you just want to run), **3** runtime + models, **4** start the worker, **5** run one edit (or **5b + 5c** to edit by correcting the transcript), **6** listen and download (MP3 for sharing; the WAV master stays in the run folder). Steps 1-4 only set things up. Nothing generates until you tick a run box yourself.

Step 1 downloads the backend code from GitHub at a fixed commit, so the notebook always installs the exact tested version. The models download separately in Step 3.

## Transcript editing: two paths

- **Step 5 (classic):** replace / insert / delete a span you select with times and checked text.
- **Steps 5b + 5c (new):** transcribe once, fix the transcript text itself, align, confirm the span by listening, preview, save.

The 5b/5c flow has no verified GPU pass yet: listen to every preview before using it. Corrected or low-confidence words always require a listening confirmation before an edit can preview.


In [ ]:
# @title Step 1. Install tools and the backend v0.6.0 (voice enhancer + transcript editor; no inference)
# Downloads the pinned backend (v0.6.0) from the GitHub repo at a fixed commit, so later repo
# changes can't change what this notebook installs. Repo: github.com/advayc/auk-production-backend.
# Qwen2.5-Omni-3B conditioner remains non-commercial.
import sys, subprocess, pathlib, os
if not (3,11) <= sys.version_info[:2] <= (3,13):
    raise RuntimeError("Use a Python 3.11-3.13 Colab runtime.")
WORK_DIR = "/content/auk-production"  # @param {type:"string"}
COMMERCIAL_USE = False  # @param {type:"boolean"}
COMMERCIAL_LICENSE_CONFIRMED = False  # @param {type:"boolean"}
if COMMERCIAL_USE and not COMMERCIAL_LICENSE_CONFIRMED: raise RuntimeError("Obtain commercial model-license clearance first.")
ROOT=pathlib.Path(WORK_DIR).resolve()
if str(ROOT) in ("/", "/content", "/tmp", str(pathlib.Path.home())): raise ValueError("Use a dedicated working directory")
ROOT.mkdir(parents=True,exist_ok=True);os.umask(0o077)
subprocess.run(["apt-get","-qq","update"],check=True)
subprocess.run(["apt-get","-qq","install","-y","ca-certificates","curl","ffmpeg","libgomp1","libssl-dev","git","cmake","build-essential","python3-venv"],check=True)
BACKEND_REPO="https://github.com/advayc/auk-production-backend.git"
BACKEND_COMMIT="3ea883e01c44e5bad5656dcab53333f7b76d3a19"  # backend v0.6.0, tested 2026-11-20
REPO_DIR=ROOT/"auk-production-backend"
if not REPO_DIR.is_dir():
    subprocess.run(["git","clone","--quiet",BACKEND_REPO,str(REPO_DIR)],check=True)
subprocess.run(["git","-C",str(REPO_DIR),"checkout","--quiet",BACKEND_COMMIT],check=True)
actual=subprocess.run(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],check=True,capture_output=True,text=True).stdout.strip()
if actual!=BACKEND_COMMIT: raise RuntimeError("Backend checkout mismatch: "+actual)
BACKEND_DIR=REPO_DIR/"backend"
import shutil; shutil.rmtree(BACKEND_DIR/"build", ignore_errors=True)  # never reuse stale build files
subprocess.run([sys.executable,"-m","pip","install","--no-deps",str(BACKEND_DIR)],check=True)
if "AUK_WORKER" in globals():AUK_WORKER.close();del AUK_WORKER
for name in list(sys.modules):
    if name=="auk_reliability" or name.startswith("auk_reliability."):del sys.modules[name]
import auk_reliability
assert auk_reliability.__version__=="0.6.0"
from auk_reliability import worker as _w, durable as _d
assert "whole_clip" in pathlib.Path(_w.__file__).read_text() and "only=None" in pathlib.Path(_d.__file__).read_text(), "Stale backend files were installed"
from auk_reliability import enhance as _e
assert "adaptive_full_wet" in pathlib.Path(_e.__file__).read_text(), "Stale backend files were installed"
from auk_reliability import transcriptflow as _tf
assert "asr_accepted" in pathlib.Path(_tf.__file__).read_text(), "Stale backend files were installed"
CHECKPOINT = None  # a checkpoint object from before this cell may point at the OLD backup; run P0 again (new backup folder)
print("Installed",auk_reliability.__version__,"from GitHub @",BACKEND_COMMIT[:12])


In [ ]:
# @title Step 2. Optional: mount Drive to save files across sessions (no generation)
ENABLE_DRIVE_BACKUP = False # @param {type:"boolean"}
RESTORE_CHECKPOINT = False # @param {type:"boolean"}
# Tick RESTORE_CHECKPOINT only on a recycled runtime. It reads RESTORE_FROM and never writes to it. BACKUP_DIR must be a different, new folder.
RESTORE_FROM = "/content/drive/MyDrive/auk-checkpoints/session-2026-10-16-v025-b" # @param {type:"string"}
BACKUP_DIR = "/content/drive/MyDrive/auk-checkpoints/session-2026-11-20-handoff" # @param {type:"string"}
DATASET_DIR = "/content/auk-public-v1" # @param {type:"string"}
RUNS_DIR = "/content/auk-runs" # @param {type:"string"}
CHECKPOINT = None
if ENABLE_DRIVE_BACKUP:
    from google.colab import drive
    from auk_reliability.durable import Checkpoint
    drive.mount('/content/drive')
    if RESTORE_CHECKPOINT:
        if __import__('os').path.realpath(RESTORE_FROM) == __import__('os').path.realpath(BACKUP_DIR): raise ValueError('BACKUP_DIR must be a different, new folder from RESTORE_FROM')
        # v0.2.5: restore only the corpus/runs from the old backup. The backend comes from P1, never from the old backup.
        old = Checkpoint(RESTORE_FROM, {"dataset":DATASET_DIR,"runs":RUNS_DIR})
        print(old.restore(only=("dataset","runs")))
        # v0.2.6: also bring back earlier request folders (hash-checked copy; skips identical files; stops if a local file differs).
        import json as _j, hashlib as _h
        _src = __import__("pathlib").Path(RESTORE_FROM)
        _recs = [r for r in _j.loads((_src/"checkpoint.json").read_text())["files"] if r["path"].startswith("backend/shared-requests/")]
        def _sha(p):
            h = _h.sha256()
            with open(p, "rb") as f:
                for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
            return h.hexdigest()
        _copied = 0
        for r in _recs:
            rel = r["path"].split("/", 1)[1]
            if ".." in rel.split("/"): raise ValueError("unsafe path")
            sp, tp = _src/r["path"], ROOT/rel
            if _sha(sp) != r["sha256"]: raise ValueError("Corrupt backup file " + r["path"])
            if tp.exists():
                if _sha(tp) != r["sha256"]: raise FileExistsError("Local file differs: " + str(tp))
                continue
            tp.parent.mkdir(parents=True, exist_ok=True); __import__("shutil").copyfile(sp, tp); _copied += 1
        print({"request_files_restored": _copied, "listed": len(_recs)})
        import glob, shutil, pathlib as _pl
        tools = sorted(glob.glob(RESTORE_FROM + "/backend/recovery-tools/*"))
        if tools:
            for f in _pl.Path(tools[-1]).glob("*.py"):
                if not (_pl.Path("/content")/f.name).exists(): shutil.copyfile(f, _pl.Path("/content")/f.name)
            print("Recovery tools copied from", tools[-1])
    CHECKPOINT = Checkpoint(BACKUP_DIR, {"backend":ROOT,"dataset":DATASET_DIR,"runs":RUNS_DIR})
    print(CHECKPOINT.sync())
else: print("Drive backup off. Nothing needed here: upload your audio in step 5 and download results in step 6. Files live only for this session. Tick ENABLE_DRIVE_BACKUP to keep dataset/runs in your Drive across sessions.")


In [ ]:
# @title Step 3. Set up the runtime and download the models (downloads, no generation)
from auk_reliability.core import Limits, gpu_details, setup_runtime, setup_models, command, atomic_json
import site, os, subprocess, sys
LIMITS=Limits();GPU=gpu_details();print(GPU)
if GPU["compute_capability"] not in ("7.5","8.9"):raise RuntimeError("The pinned prebuilt serves T4 (7.5) and L4 (8.9) only. Other GPUs require the explicit source-build path.")
if GPU["compute_capability"]=="8.9":print("L4 (SM89) prebuilt: on-VM qualification = ldd + --help + this session's QA-gated edits.")
if GPU["total_mib"]<14000:raise RuntimeError("At least roughly 16 GiB total VRAM required")
# audio.cpp SM75 release uses CUDA 12.8, including on Colab images that ship CUDA 13.
subprocess.run([sys.executable,"-m","pip","install","--quiet","nvidia-cublas-cu12==12.8.4.1","nvidia-cuda-runtime-cu12==12.8.90","nvidia-cufft-cu12==11.3.3.83"],check=True)
search=set(site.getsitepackages()+[site.getusersitepackages()]);libdirs=[]
for root in search:
    for component in ("cublas","cuda_runtime","cufft"):
        p=pathlib.Path(root)/"nvidia"/component/"lib"
        if p.is_dir():libdirs.append(str(p))
os.environ["LD_LIBRARY_PATH"]=":".join(libdirs+[os.environ.get("LD_LIBRARY_PATH","")])
MEMORY_SAVER=True
BINARY,RUNTIME_LOCK=setup_runtime(ROOT,mode="t4_prebuilt",gpu=GPU)
link_check=command(["ldd",str(BINARY)],timeout=30)
if "not found" in link_check:raise RuntimeError("Runtime has unresolved libraries: "+link_check)
MODEL_DIR,MODEL_LOCK=setup_models(ROOT,["base_f32","base_q8","flash_f16"])
PROVENANCE={"runtime":RUNTIME_LOCK,"model":MODEL_LOCK,"environment":{"python":sys.version,"gpu":GPU,"nvidia_smi":command(["nvidia-smi"]),"ffmpeg":command(["ffmpeg","-version"]).splitlines()[0],"backend_commit":BACKEND_COMMIT}}
atomic_json(ROOT/"environment.json",PROVENANCE)
print("Ready. No model generation started. First request includes cold startup.")


In [ ]:
# @title Step 4. Start the worker and the QA checks (downloads, no generation)
import pathlib
ROOT = pathlib.Path(ROOT) if isinstance(ROOT, str) else ROOT
ASR_LANGUAGE = "en"  # @param {type:"string"}
SETUP_SHARED_WORKER = True  # @param {type:"boolean"}
ENABLE_SPEAKER_QA = True  # @param {type:"boolean"}
GPU_HOURLY_RATE_USD = -1.0  # @param {type:"number"}
if SETUP_SHARED_WORKER:
    _need = [n for n in ("ROOT","LIMITS","GPU","MEMORY_SAVER","ASR_LANGUAGE","BINARY","MODEL_DIR") if n not in globals()]
    if _need:
        raise RuntimeError("Run steps 1 and 3 first (missing " + ", ".join(_need) + "). ")
    if "PROVENANCE" not in globals():
        PROVENANCE = {"runtime": RUNTIME_LOCK, "model": MODEL_LOCK, "environment": {"python":sys.version}}
    if "auk_reliability" not in sys.modules:
        raise RuntimeError("Run P1 first.")
    from auk_reliability.core import LocalAuKServer, setup_asr
    from auk_reliability.colab import setup_optional_speaker_qa
    from auk_reliability.qa import SemanticQAGate
    from auk_reliability.worker import ReliabilityWorker, configure_default_worker, edit_audio
    from auk_reliability.acceptance import configure_acceptance_suite, run_acceptance_suite, benchmark_profiles, recommend_profiles
    if "AUK_WORKER" in globals(): AUK_WORKER.close()
    shared_asr, shared_asr_fingerprint = setup_asr(pathlib.Path(ROOT) / "shared-qa", LIMITS, ASR_LANGUAGE)
    speaker_callback, speaker_fingerprint, speaker_setup = setup_optional_speaker_qa(
        pathlib.Path(ROOT) / "shared-speaker-qa", enabled=ENABLE_SPEAKER_QA)
    print("Speaker QA:", speaker_setup)
    if speaker_callback is None:
        raise RuntimeError("Speaker QA unavailable. Fix setup before reliability runs; unavailable identity QA never passes.")
    transport = LocalAuKServer(BINARY, MODEL_DIR, ROOT / "shared-runtime", LIMITS, MEMORY_SAVER)
    runtime_profile = {**PROVENANCE, "gpu": GPU, "simulation": False,
        "asr_fingerprint": shared_asr_fingerprint, "speaker_fingerprint": speaker_fingerprint, "speaker_setup": speaker_setup}
    if GPU_HOURLY_RATE_USD >= 0: runtime_profile["gpu_hourly_rate_usd"] = GPU_HOURLY_RATE_USD
    AUK_WORKER = ReliabilityWorker(transport, ROOT / "shared-requests",
        semantic_gate=SemanticQAGate(shared_asr, speaker_callback), runtime_profile=runtime_profile)
    configure_default_worker(AUK_WORKER)
    print("Shared worker configured. No generation started. Attribute QA still needs human review.")
else:
    print("Shared worker setup is off. Enable when ready for QA downloads/install.")


In [ ]:
# @title Step 5. Run one edit (tick RUN_EDIT yourself; nothing runs on Run all)
INPUT_AUDIO = ""  # @param {type:"string"}
RUN_EDIT = False  # @param {type:"boolean"}
MODE = "voice_enhancer"  # @param ["voice_enhancer", "transcript_editor", "voice_editor"]
# voice_enhancer (cleanup): enhance | denoise | dereverb
# transcript_editor: replace | insert | delete
# (classic span edit with checked segment text; to correct the transcript itself, use steps 5b + 5c)
# voice_editor (prompt edits): emotion | voice_characteristics | whisper | speed | custom
OPERATION = "enhance"  # @param ["enhance", "denoise", "dereverb", "replace", "insert", "delete", "emotion", "voice_characteristics", "whisper", "speed", "custom"]
PROFILE = "base_f32"  # @param ["base_f32", "base_q8", "flash_f16"]
RIGHTS_CONFIRMED = False  # @param {type:"boolean"}
SEED = 42  # @param {type:"integer"}
# Human-checked transcript of the recording. Every mode uses it for the automatic content check.
CHECKED_SOURCE_TEXT = ""  # @param {type:"string"}
# whole_clip: edit the whole <=15 s recording, no segment times needed.
# localized: your own checked segment times below.
TRANSCRIPT_SCOPE = "whole_clip"  # @param ["whole_clip", "localized"]
# Per-operation settings. Edit only the ones for your chosen OPERATION.
EMOTION = "calm"  # @param {type:"string"}
VOICE_DESCRIPTION = "warm, slightly deeper voice"  # @param {type:"string"}
SPEED_FACTOR = 1.25  # @param {type:"number"}
CUSTOM_INSTRUCTION = "Keep every word but sound more confident."  # @param {type:"string"}
# Transcript edits need a listened-to, nonempty interval (max 15 s) and the exact checked text inside it.
SEGMENT_START_SECONDS = 1.23  # @param {type:"number"}
SEGMENT_END_SECONDS = 2.05  # @param {type:"number"}
SEGMENT_TEXT = "the better team"  # @param {type:"string"}
# replace: phrase to replace / delete: phrase to remove / insert: text to add. Type it WITHOUT quote marks.
EDIT_TARGET_TEXT = ""  # @param {type:"string"}
# replace only. Type it WITHOUT quote marks.
EDIT_REPLACEMENT_TEXT = ""  # @param {type:"string"}
INSERT_ANCHOR = ""  # @param {type:"string"}
INSERT_POSITION = "before"  # @param ["before", "after"]

def _clean(v): return "" if v is None else str(v).strip().strip("\"'\u201c\u201d").strip()

def build_parameters():
    global EDIT_TARGET_TEXT, EDIT_REPLACEMENT_TEXT
    EDIT_TARGET_TEXT, EDIT_REPLACEMENT_TEXT = _clean(EDIT_TARGET_TEXT), _clean(EDIT_REPLACEMENT_TEXT)
    base = {"source_text": CHECKED_SOURCE_TEXT, "rights_confirmed": RIGHTS_CONFIRMED,
            "profile": PROFILE, "seed": SEED}
    op = OPERATION
    if MODE == "voice_enhancer":
        if op not in ("enhance", "denoise", "dereverb"):
            raise RuntimeError(f"{op} is not an enhancer operation. Use enhance, denoise or dereverb.")
        return base
    if MODE == "voice_editor":
        if op not in ("emotion", "voice_characteristics", "whisper", "speed", "custom"):
            raise RuntimeError(f"{op} is not a prompt-editing operation. Check the OPERATION list above.")
        if op == "emotion": base["emotion"] = EMOTION
        elif op == "voice_characteristics": base["description"] = VOICE_DESCRIPTION
        elif op == "speed": base["speed_factor"] = SPEED_FACTOR
        elif op == "custom":
            base["instruction"] = CUSTOM_INSTRUCTION
            base["semantic_contract"] = {"preserve_transcript": True}
        return base
    if op not in ("replace", "insert", "delete"):
        raise RuntimeError(f"{op} is not a transcript operation. Check the OPERATION list above.")
    if not SEGMENT_TEXT.strip() or not EDIT_TARGET_TEXT.strip():
        raise RuntimeError("Transcript edits need SEGMENT_TEXT and EDIT_TARGET_TEXT (checked by listening).")
    if TRANSCRIPT_SCOPE == "whole_clip":
        seg = CHECKED_SOURCE_TEXT
        base.update(segment_start_seconds=0, segment_end_seconds=None, segment_text=seg, segment_scope="whole_clip")
    else:
        seg = SEGMENT_TEXT
        base.update(segment_start_seconds=SEGMENT_START_SECONDS, segment_end_seconds=SEGMENT_END_SECONDS,
                    segment_text=seg, alignment_verified=True)
    if op == "replace":
        if not EDIT_REPLACEMENT_TEXT.strip(): raise RuntimeError("Set EDIT_REPLACEMENT_TEXT.")
        base.update(original_text=EDIT_TARGET_TEXT, replacement_text=EDIT_REPLACEMENT_TEXT,
                    edited_segment_text=seg.replace(EDIT_TARGET_TEXT, EDIT_REPLACEMENT_TEXT, 1))
    elif op == "delete":
        base.update(text=EDIT_TARGET_TEXT,
                    edited_segment_text=" ".join(seg.replace(EDIT_TARGET_TEXT, "", 1).split()))
    else:
        if not INSERT_ANCHOR.strip(): raise RuntimeError("Set INSERT_ANCHOR (a phrase that occurs once in SEGMENT_TEXT).")
        base.update(text=EDIT_TARGET_TEXT, anchor=INSERT_ANCHOR, position=INSERT_POSITION,
                    edited_segment_text=(EDIT_TARGET_TEXT + " " + INSERT_ANCHOR) if INSERT_POSITION == "before" else (INSERT_ANCHOR + " " + EDIT_TARGET_TEXT))
        base["edited_segment_text"] = seg.replace(INSERT_ANCHOR, base["edited_segment_text"], 1)
    return base

if RUN_EDIT:
    if "CHECKPOINT" not in globals(): raise RuntimeError("Run step 2 first.")
    if "AUK_WORKER" not in globals(): raise RuntimeError("Run step 4 first.")
    if not RIGHTS_CONFIRMED: raise RuntimeError("Tick RIGHTS_CONFIRMED only for recordings you have rights to process.")
    if not CHECKED_SOURCE_TEXT.strip(): raise RuntimeError("Set CHECKED_SOURCE_TEXT (your human-checked transcript of the recording).")
    if INPUT_AUDIO.strip():
        SOURCE_FILE = pathlib.Path(INPUT_AUDIO)
        if not SOURCE_FILE.is_file(): raise RuntimeError(f"No file at {SOURCE_FILE}. Leave INPUT_AUDIO empty to upload instead.")
    else:
        from google.colab import files as _files
        print("Upload one recording to edit:")
        _up = _files.upload()
        if not _up: raise RuntimeError("No file uploaded.")
        SOURCE_FILE = pathlib.Path("/content") / next(iter(_up))
    PARAMETERS = build_parameters()
    print("ABOUT TO RUN (check before it uses the GPU):", MODE, "/", OPERATION, "| scope:", TRANSCRIPT_SCOPE if MODE == "transcript_editor" else "n/a")
    print("  source file:", SOURCE_FILE.name)
    for _k in ("source_text", "original_text", "replacement_text", "text", "anchor", "position", "edited_segment_text"):
        if _k in PARAMETERS: print(f"  {_k}: {PARAMETERS[_k]!r}")
    try:
        result = edit_audio(SOURCE_FILE, MODE, OPERATION, PARAMETERS)
    finally:
        if CHECKPOINT: print(CHECKPOINT.sync())
    print("RESULT:", result.outcome, "| reason:", result.reason, "| technical_success:", result.technical_success,
          "| automatic_qa_pass:", result.automatic_qa_pass, "| latency_s:", round(result.latency_seconds))
    print("request_dir:", result.request_dir, "(paste this into step 6 to listen again later)")
    for _a in result.attempts:
        _t = (_a.get("qa") or {}).get("checks", {}).get("transcript", {})
        _sp = (_a.get("qa") or {}).get("checks", {}).get("speaker", {})
        print(f"  attempt {_a['attempt']} seed {_a['seed']}: ASR error {_t.get('error_rate')} | speaker {_sp.get('score')} | heard:", repr(_t.get("observed")))
    if result.output_path:
        from IPython.display import Audio, display
        display(Audio(filename=result.output_path))
    print("Human requested-effect/naturalness/seam review remains required.")
else:
    print("Nothing ran. Pick MODE + OPERATION, set CHECKED_SOURCE_TEXT, tick RIGHTS_CONFIRMED and RUN_EDIT, then run this cell.")


In [ ]:
# @title Step 5b. Correct the transcript and pick the words to edit (new flow; no generation)
TS_INPUT_AUDIO = ""  # @param {type:"string"}
RUN_TRANSCRIBE = False  # @param {type:"boolean"}
USE_ALIGNER = True  # @param {type:"boolean"}
# Leave CORRECTED_TEXT empty to accept the transcription as-is. Otherwise type the full corrected sentence.
CORRECTED_TEXT = ""  # @param {type:"string"}
# The exact words to edit (as they appear in the final transcript), and what they should become.
TS_SPAN_TEXT = ""  # @param {type:"string"}
TS_EDITED_SPAN_TEXT = ""  # @param {type:"string"}
if not RUN_TRANSCRIBE:
    print("Tick RUN_TRANSCRIBE to transcribe and review. Nothing runs on Run all.")
else:
    import subprocess, pathlib
    from IPython.display import Audio, display
    for _n in ("ROOT", "LIMITS", "AUK_WORKER", "shared_asr"):
        if _n not in globals(): raise RuntimeError("Run steps 1-4 first (missing " + _n + ").")
    _src = TS_INPUT_AUDIO.strip()
    if not _src:
        from google.colab import files as _files
        print("Upload one recording to edit:")
        _up = _files.upload()
        if not _up: raise RuntimeError("No file uploaded.")
        _src = next(iter(_up))
    _sess_root = pathlib.Path(ROOT) / "transcript-sessions"
    _sess_root.mkdir(exist_ok=True)
    # Review slices need PCM16 24 kHz: normalize a session copy; the original file is untouched.
    _norm = _sess_root / "session-source.wav"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", _src, "-ac", "1", "-ar", "24000", "-sample_fmt", "s16", str(_norm)], check=True)
    _aligner = None
    if USE_ALIGNER:
        from auk_reliability.forcedalign import setup_aligner
        _aligner, _afp = setup_aligner(pathlib.Path(ROOT), LIMITS)  # first run installs the pinned CPU aligner stack
        print("Aligner fingerprint:", _afp)
    from auk_reliability.transcriptflow import TranscriptSession
    TS_SESSION = TranscriptSession(_norm, shared_asr, _sess_root / "session", aligner=_aligner, flow_registry=AUK_WORKER.flow_registry)
    TS_SESSION.transcribe()
    print("Transcription:", TS_SESSION.text)
    if CORRECTED_TEXT.strip():
        TS_SESSION.correct(CORRECTED_TEXT.strip())
        print("Corrected:", TS_SESSION.text)
    _flagged_words = sorted({w["word"] for w in TS_SESSION.transcript() if w["needs_review"]})
    if _flagged_words: print("Words flagged for listening review:", ", ".join(_flagged_words))
    TS_MANIFEST = None
    if TS_SPAN_TEXT.strip():
        _flagged = TS_SESSION.span_needs_review(TS_SPAN_TEXT.strip())
        TS_MANIFEST = TS_SESSION.review_candidates(TS_SPAN_TEXT.strip())
        if not _flagged:
            print("Span boundaries are measured; step 5c previews without a listening confirmation.")
        else:
            print("This span needs a listening confirmation for:", ", ".join(_flagged))
        print("Candidate span slices (listen, then set CANDIDATE_INDEX in step 5c):")
        for _i, _m in enumerate(TS_MANIFEST):
            print("[{}] {:.2f}s - {:.2f}s".format(_i, _m["start"], _m["end"]))
            display(Audio(filename=_m["path"]))
    print("Session ready. Finish in step 5c (same runtime).")


In [ ]:
# @title Step 5c. Preview the transcript edit, listen, then save (generation)
CANDIDATE_INDEX = 0  # @param {type:"integer"}
# Tick only after you actually listened to the candidate slices from step 5b and picked CANDIDATE_INDEX.
LISTENED_CONFIRMED = False  # @param {type:"boolean"}
RUN_PREVIEW = False  # @param {type:"boolean"}
# Saving is a separate run AFTER listening to the preview: paste its preview_id, untick RUN_PREVIEW, tick SAVE_OUTPUT.
PREVIEW_ID = ""  # @param {type:"string"}
SAVE_OUTPUT = False  # @param {type:"boolean"}
OUTPUT_NAME = "transcript-edit.wav"  # @param {type:"string"}
if not RUN_PREVIEW and not SAVE_OUTPUT:
    print("Tick RUN_PREVIEW to generate one preview. Nothing runs on Run all.")
elif SAVE_OUTPUT and RUN_PREVIEW:
    raise RuntimeError("A preview generated in this run has not been heard yet. To save: listen, then re-run with RUN_PREVIEW unticked, PREVIEW_ID pasted, SAVE_OUTPUT ticked.")
elif SAVE_OUTPUT:
    for _n in ("TS_SESSION",):
        if _n not in globals(): raise RuntimeError("Run steps 1-4 and 5b first (missing " + _n + ").")
    if not PREVIEW_ID.strip(): raise RuntimeError("Paste the preview_id printed by the preview run into PREVIEW_ID.")
    import pathlib
    _dest = pathlib.Path(RUNS_DIR) / "transcript-edits" / OUTPUT_NAME
    print(TS_SESSION.finalize(PREVIEW_ID.strip(), _dest))
    if CHECKPOINT: print(CHECKPOINT.sync())
else:
    import pathlib
    from IPython.display import Audio, display
    for _n in ("TS_SESSION", "AUK_WORKER"):
        if _n not in globals(): raise RuntimeError("Run steps 1-4 and 5b first (missing " + _n + ").")
    if not TS_SPAN_TEXT.strip() or not TS_EDITED_SPAN_TEXT.strip():
        raise RuntimeError("Set TS_SPAN_TEXT and TS_EDITED_SPAN_TEXT in step 5b first.")
    _flagged = TS_SESSION.span_needs_review(TS_SPAN_TEXT.strip())
    _token = None
    if _flagged:
        if not LISTENED_CONFIRMED:
            raise RuntimeError("This span needs a listening confirmation for: " + ", ".join(_flagged) +
                               ". Listen to the candidate slices in step 5b, set CANDIDATE_INDEX, then tick LISTENED_CONFIRMED.")
        if TS_MANIFEST is None: raise RuntimeError("Re-run step 5b to render the review candidates.")
        _entry = TS_MANIFEST[CANDIDATE_INDEX]
        _token = TS_SESSION.confirm_span(_entry)
        print("Confirmed candidate [{}] {:.2f}s - {:.2f}s for: {}".format(CANDIDATE_INDEX, _entry["start"], _entry["end"], ", ".join(_flagged)))
    _params = TS_SESSION.span_params(TS_SPAN_TEXT.strip(), TS_EDITED_SPAN_TEXT.strip(), operation="replace", confirmation_token=_token)
    _result = TS_SESSION.preview(AUK_WORKER.edit_audio, "replace", _params)
    print("Technical:", _result["technical_success"], "auto QA:", _result["automatic_qa_pass"])
    if _result["output_path"]:
        display(Audio(filename=_result["output_path"]))
    print("preview_id:", _result["preview_id"])
    print("Listen before using: this flow has no verified GPU pass yet. To save, re-run with RUN_PREVIEW unticked, PREVIEW_ID pasted, SAVE_OUTPUT ticked.")


In [ ]:
# @title Step 6. Listen again, save a review, or download (no inference)
REQUEST_DIR = ""  # @param {type:"string"}
REVIEWER = ""  # @param {type:"string"}
SAVE_REVIEW = False  # @param {type:"boolean"}
QUALITY_PASSED = False  # @param {type:"boolean"}
REVIEW_NOTES = ""  # @param {type:"string"}
if REQUEST_DIR.strip():
    import json
    from IPython.display import Audio, display
    folder=pathlib.Path(REQUEST_DIR);result=json.loads((folder/"result.json").read_text())
    print("Technical:",result["technical_success"],"auto QA:",result["automatic_qa_pass"],"reason:",result["reason"])
    if (folder/"input.wav").is_file():display(Audio(filename=str(folder/"input.wav")))
    if result["output_path"]:display(Audio(filename=result["output_path"]))
    if result["output_path"]:
        # MP3 for sharing, encoded from the WAV master. The WAV stays the master file; MP3 is lossy, never call it lossless.
        import subprocess as _sp, json as _js
        def _lu(p):
            r=_sp.run(["ffmpeg","-hide_banner","-nostats","-i",str(p),"-af","loudnorm=print_format=json","-f","null","-"],capture_output=True,text=True)
            m=_js.loads(r.stderr[r.stderr.rindex("{"):r.stderr.rindex("}")+1]);return float(m["input_i"])
        wav=pathlib.Path(result["output_path"]);mp3=wav.with_suffix(".mp3")
        def _enc(extra=[]):
            _sp.run(["ffmpeg","-hide_banner","-v","error","-nostdin","-xerror","-i",str(wav)]+extra+["-c:a","libmp3lame","-b:a","320k","-y",str(mp3)],check=True)
        wi=_lu(wav);_enc();mi=_lu(mp3)
        if abs(mi-wi)>0.2:  # codec level shift: re-encode with compensation, then re-check the encoded file
            _enc(["-af",f"volume={wi-mi:+.2f}dB"]);mi=_lu(mp3)
        print(f"MP3 (320 kbps): {mp3.name} | level check on the encoded file: WAV {wi:.2f} LUFS, MP3 {mi:.2f} LUFS")
        from google.colab import files as _f;_f.download(str(mp3))
    print("Check words, target effect, identity where preserved, speed/timing, naturalness, seams and endings. Review every selected benchmark/acceptance output.")
    if SAVE_REVIEW:
        from auk_reliability.workflow import save_review
        print(save_review(folder,reviewer=REVIEWER,passed=QUALITY_PASSED,notes=REVIEW_NOTES))
else:print("Paste the request_dir printed by step 5 into REQUEST_DIR above.")

if SAVE_REVIEW and CHECKPOINT:print(CHECKPOINT.sync())


## Done? Shut it down

GPU billing only ends with **Runtime > Disconnect and delete runtime** (then check **Manage sessions** shows none). Changing runtime type while disconnected starts a new one.

The editing checks (output validation, bounded retries, transcript and speaker QA) run inside the backend on every request: a failed gate produces no output rather than a bad one.